# Feasibility of EODAG

In [1]:
import helper.maps as helper 
import helper.harvester_code as harvester
import json
import pystac_client

from eodag import EODataAccessGateway
dag = EODataAccessGateway()

### Copernicus Dataspace Ecosystem

#### 1. Searching Data with OData API

In [2]:
#url = "https://datahub.creodias.eu/odata/v1"
url = "https://catalogue.dataspace.copernicus.eu/odata/v1"
limit: int = 1000
start_time = "2026-04-29T15:00:00.000Z"
end_time = "2026-04-29T16:00:00.000Z"

filter_base = f"(PublicationDate ge {start_time} and PublicationDate lt {end_time}) and Online eq true"

#filter_s1_slc = "(startswith(Name,'S1') and contains(Name,'SLC') and not contains(Name,'_COG') and not contains(Name, 'CARD_BS'))"
#filter_s1_grd = "(startswith(Name,'S1') and contains(Name,'GRD') and not contains(Name,'_COG') and not contains(Name, 'CARD_BS'))"
filter_s2_l2a = "(startswith(Name,'S2') and (contains(Name,'L2A')) and not contains(Name,'_N9999') and not contains(Name,'_N0500'))"
#filter_s2_l1c = "(startswith(Name,'S2') and (contains(Name,'L1C')) and not contains(Name,'_N9999') and not contains(Name,'_N0500'))"
#filter_s3 = "(startswith(Name,'S3') and not contains(Name,'NRTI_'))"
#filter_s5p = "(startswith(Name,'S5P') and not contains(Name,'NRTI_'))"

filters = [f"({filter_base}) and ({filter_s2_l2a})"]

odata_scenes = harvester.search(
    api_url=url,
    max_items=limit,
    filters=filters,
)

Executing OData query: https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=((PublicationDate ge 2026-04-29T15:00:00.000Z and PublicationDate lt 2026-04-29T16:00:00.000Z) and Online eq true) and ((startswith(Name,'S2') and (contains(Name,'L2A')) and not contains(Name,'_N9999') and not contains(Name,'_N0500')))&$top=1000
Found 654 scenes
Result set complete


#### 2. Searching Data with eodag
We cannot use the `start` and `end` parameters of the `search()` function since we want to filter by publication time instead of sensing time.

In [3]:
eodag_products = dag.search(
    provider="cop_dataspace",
    collection="S2_MSI_L2A",
    #start=start_time,
    #end=end_time,
    #geom="POLYGON((3 55, 3 47, 18 47, 18 55, 3 55))",
    published_after=start_time,
    published_before=end_time,
    limit=limit
)
print(f"Found {len(eodag_products.data)} products")

Found 654 products


#### 3. Compare search results
Check for each OData result if eodag has found the same product.

In [4]:
matching_product_uid = []
for odata_scene in odata_scenes:
    uid = odata_scene["uid"]
    for eodag_product in eodag_products:
        if eodag_product.properties["uid"] == uid:
            matching_product_uid.append(uid)            

print(f"{len(odata_scenes)} OData products found")
print(f"{len(eodag_products)} eodag products found")
print(f"{len(matching_product_uid)} Matching products")

654 OData products found
654 eodag products found
654 Matching products


In [ ]:
folium_map_cop = helper.map_results(False, eodag_products)
folium_map_cop

#### 4. Downloading data with eodag

In [ ]:
paths = dag.download(
    eodag_products[1],
    output_dir=r"/dss/dsshome1/0C/di54wir/Daten/harvester_eodag",
)

### USGS

#### 1. Searching Data at STAC API

In [6]:
usgs_stac_url = "https://landsatlook.usgs.gov/stac-server"
collection = "landsat-c2l2-sr"
limit = 1000
start_time = "2026-04-29T20:00:00.000Z"
end_time = "2026-04-29T21:00:00.000Z"
#bbox = "8,40,18,60".split(",")
bbox = None
stacql_query = {
    "created": {"gte": start_time, "lt": end_time},
}

print(f"Collection: {collection}")
print(f"BBox: {bbox}")
print(f"STAC Query: {json.dumps(stacql_query, indent=4)}")
print(f"Limit: {limit}\n")

client = pystac_client.Client.open(usgs_stac_url)
usgs_items = client.search(
    collections=[collection],
    bbox=bbox,
    datetime=None,
    max_items=limit,
    query=stacql_query
).item_collection()

print(f"Found {len(usgs_items)} products")

Collection: landsat-c2l2-sr
BBox: None
STAC Query: {
    "created": {
        "gte": "2026-04-29T20:00:00.000Z",
        "lt": "2026-04-29T21:00:00.000Z"
    }
}
Limit: 1000

Found 121 products


#### 2. Searching Data with eodag
We cannot use the `start` and `end` parameters of the `search()` function since we want to filter by publication time instead of sensing time.

In [ ]:
usgs_products = dag.search(
    provider="usgs",
    collection="TBD: eodag type for landsat-c2l2-sr",
    bbox=bbox,
    #start="2018-06-01",
    #end="2019-06-15",
    limit=limit,
    #TBD: find the param representing the 'created' timestamp
)

print(f"Found {len(usgs_products.data)} products")

#### 3. Compare search results
Check for each USGS STAC result if eodag has found the same product.

In [ ]:
# Since STAC is used in both cases we can compare the item ids e.g. usgs_items[0].id
# TBD

In [ ]:
folium_map_usgs = helper.map_results(True, usgs_products)
folium_map_usgs

#### 4. Download the Data

In [ ]:
paths = dag.download(
    usgs_products[3],
    output_dir=r"/dss/dsshome1/0C/di54wir/Daten/harvester_eodag",
)